# From Microbiomes to Predictions: Genomics and AI in Animal Science

This notebook walks through a complete example of how genomic data from microbial communities can be used, with machine learning, to predict an animal phenotype — and how we can explain the model's reasoning. We will work with **rumen microbiome** samples from cattle and ask whether we can predict which animals are low or high methane emitters.

**You do not need to understand or modify any code in this notebook. Run each cell and read the outputs.**

**How to run:** Press `Shift+Enter` to run the currently selected cell, or use `Runtime > Run all` in the Colab menu to run the entire notebook from top to bottom.

Sections of this notebook reveal themselves as you go: each cell's explanation and result only appear once you've run the cell before it, so try not to skip ahead.


## Section 1: The Biology — What Are We Looking At?

**Ruminants** (cattle, sheep, goats) digest tough plant material using a specialised stomach chamber called the rumen, which hosts a dense community of microbes. A by-product of this microbial fermentation is **methane**, a potent greenhouse gas — so understanding what drives methane emissions in livestock matters for both agriculture and climate change.

The community of microbes living in an animal's gut is called its **microbiome**. Different animals host different microbial communities, and there is growing evidence that the composition of the rumen microbiome is linked to how much methane an animal produces.

To study this with computers, we need a way to turn a microbial community into numbers. One approach is to count characteristic short DNA sequence patterns, called **k-mers**, and group similar patterns together into a **genomic embedding** — a vector of numbers that summarises the microbial makeup of a sample.

This is the bridge between biology and machine learning.


### A concrete example: what does a k-mer actually look like?

A **k-mer** is simply a short substring of length *k* taken from a longer DNA sequence, which is itself just a string of the letters A, C, G and T. To count the k-mers in a sequence, we slide a window of length *k* along it, one letter at a time, and tally how often each pattern appears. Do this across millions of DNA reads from a sample, and the resulting counts start to characterise which microbes — and how much of each — are present.

**What to expect:** a short made-up DNA sequence, and a table of its most common 4-letter k-mers and how many times each occurs. This is the raw ingredient behind the "genomic feature" columns you'll see throughout this notebook — though, as explained below, there's one more step between this table and those columns.

**▶ Run the cell below**


### Step 1: See a k-mer in action

In [ ]:
# Workshop notebook: "From Microbiomes to Predictions: Genomics and AI in Animal Science"
# Prof Chris Creevey, Queen's University Belfast
# Run all cells in order. No modifications required.
# All data is synthetic and generated within this notebook.

import collections

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, precision_recall_fscore_support
from IPython.display import display, Markdown

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')

LOW_COLOR = '#2196F3'
HIGH_COLOR = '#F44336'

# A short illustrative DNA sequence (for teaching purposes only — not connected to the
# synthetic dataset used later in this notebook).
example_sequence = 'ATCGGCTAGCATCGGCTTAGGATCGGCTAGCTTAGGATCGGCATCGTTAGGCATCGGCTA'
k = 4

kmer_counts = collections.Counter(example_sequence[i:i + k] for i in range(len(example_sequence) - k + 1))
kmer_df = pd.DataFrame(kmer_counts.most_common(10), columns=[f'{k}-mer', 'Count'])
display(kmer_df)

display(Markdown('''In a real rumen microbiome sample, this process runs on millions of DNA reads rather than one 60-letter sequence, producing thousands of distinct k-mer counts per animal — far too many columns to work with directly, and mostly redundant with one another. That huge k-mer count table is what gets compressed into the much smaller **genomic embedding**: a set of derived numbers (30, in this notebook) that summarise the same underlying information. Those 30 derived numbers — not the raw k-mer counts themselves — are the "genomic feature" columns used for the rest of this notebook.

> **Discussion:** Before we look at the data, what do you think the microbiome of a low methane cow might look like compared to a high methane cow? What types of microbes might be different?

### How diverse is each animal's microbiome?

Every sample contains dozens of different microbial signals, but usually only a handful dominate any one animal. To get a feel for this diversity, we can turn each sample's genomic feature values into proportions — how much of that animal's profile is made up of its most common microbial groups — and stack them into a single bar per animal. The 8 largest groups (by average proportion across the dataset) are shown individually; everything else is lumped into **Other**.

**What to expect:** two stacked bar charts, one for high emitters and one for low emitters. Each vertical bar is one animal, and the coloured segments show the proportion of each microbial group present in it. You should see broadly similar mixes across animals, dominated by the same handful of groups, with the exact proportions varying from animal to animal. That variation is normal biological diversity — and part of why finding the phenotype signal isn't as simple as eyeballing composition.

**▶ Run the cell below**'''))


### Step 2: Explore microbiome diversity

In [ ]:
N = 120          # total samples
M = 30           # genomic features
N_TRAIN_FRAC = 80 / 120

# Named genomic features. The first 10 carry biological signal;
# the remaining 20 are generic/noise features.
feature_names = [
    'Butyrivibrio_cluster',        # cellulolytic, competes with methanogens -> LOW emitters
    'Methanobrevibacter_cluster',  # primary methanogen -> HIGH emitters
    'Ruminococcus_cluster',        # fibre degrader, competes with methanogens -> LOW emitters
    'Prevotella_cluster',          # generalist fermenter, moderate -> LOW emitters
    'Fibrobacter_cluster',         # cellulolytic, competes with methanogens -> LOW emitters
    'Methanomassiliicoccus_cluster', # secondary methanogen -> HIGH emitters
    'Lachnospiraceae_cluster',     # noise
    'Treponema_cluster',           # noise
    'Succiniclasticum_cluster',    # noise
    'Clostridium_cluster',         # noise
] + [f'genomic_feature_{i}' for i in range(11, 31)]

assert len(feature_names) == M

# Phenotype labels: ~60/40 low/high split
n_low = int(round(N * 0.6))
n_high = N - n_low
labels = np.array([0] * n_low + [1] * n_high)  # 0 = low_emitter, 1 = high_emitter
np.random.shuffle(labels)

# Continuous phenotype: methane emissions (g/day)
methane = np.where(
    labels == 0,
    np.random.normal(180, 25, size=N),
    np.random.normal(310, 35, size=N),
)

# Build genomic feature matrix.
# Biological rationale for each signal:
#   Butyrivibrio, Fibrobacter, Ruminococcus: efficient fibre-fermenting bacteria that
#       outcompete methanogens for hydrogen -> elevated in LOW emitters.
#   Methanobrevibacter, Methanomassiliicoccus: methanogenic archaea that directly
#       produce methane -> elevated in HIGH emitters.
#   Prevotella: generalist fermenter loosely associated with more efficient,
#       lower-emission fermentation pathways -> mildly elevated in LOW emitters.
#   All remaining features: no true phenotype association, included as biological
#       and technical noise (as in real microbiome data).
X = np.random.normal(loc=0.0, scale=1.0, size=(N, M))

low_boost = {'Butyrivibrio_cluster': 1.4, 'Fibrobacter_cluster': 1.2, 'Ruminococcus_cluster': 1.3,
             'Prevotella_cluster': 0.6}
high_boost = {'Methanobrevibacter_cluster': 1.6, 'Methanomassiliicoccus_cluster': 1.0}

for name, boost in low_boost.items():
    idx = feature_names.index(name)
    X[labels == 0, idx] += boost

for name, boost in high_boost.items():
    idx = feature_names.index(name)
    X[labels == 1, idx] += boost

# Turn genomic feature values into a "community composition" for this plot only:
# convert to positive weights and normalise each sample to sum to 1, giving a
# proportion per microbial group, similar to relative-abundance profiles in real
# microbiome studies. The 8 groups with the largest average proportion are shown
# individually; the rest are pooled into "Other".
composition_weights = np.exp(X)
composition_proportions = composition_weights / composition_weights.sum(axis=1, keepdims=True)

top_n = 8
mean_proportion = composition_proportions.mean(axis=0)
top_idx = np.argsort(mean_proportion)[::-1][:top_n]
other_idx = np.array([i for i in range(M) if i not in top_idx])

composition_labels = [feature_names[i] for i in top_idx] + ['Other']
composition_matrix = np.column_stack(
    [composition_proportions[:, i] for i in top_idx] + [composition_proportions[:, other_idx].sum(axis=1)]
)

group_colors = list(plt.get_cmap('tab10').colors[:top_n]) + ['#B0BEC5']

fig, axes = plt.subplots(2, 1, figsize=(10, 9), dpi=100)

for ax, phenotype_val, phenotype_name in [(axes[0], 1, 'High Emitters'), (axes[1], 0, 'Low Emitters')]:
    mask = labels == phenotype_val
    comp = composition_matrix[mask]
    x_pos = np.arange(comp.shape[0])
    bottom = np.zeros(comp.shape[0])
    for j, group_label in enumerate(composition_labels):
        ax.bar(x_pos, comp[:, j], bottom=bottom, width=1.0, color=group_colors[j],
               label=group_label if ax is axes[0] else None)
        bottom += comp[:, j]
    ax.set_title(f'Microbial Community Composition: {phenotype_name} (n={mask.sum()})')
    ax.set_xlabel('Sample')
    ax.set_ylabel('Proportion of Community')
    ax.set_xlim(-0.5, comp.shape[0] - 0.5)
    ax.set_ylim(0, 1)

axes[0].legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8, title='Microbial Group')
plt.tight_layout()
plt.show()

display(Markdown('''Look closely and you may notice the high-emitter chart leaning more heavily on some colours than the low-emitter chart does — but every animal is still a different mix, and both charts share the same broad palette of groups. Any consistent difference between phenotypes is a shift in proportions, not a clean separation — which is exactly what the rest of this notebook is designed to uncover.

### A first look: average abundance by phenotype

The composition charts above hinted at a subtle shift between phenotypes, but with so many overlapping colours in each bar, it's hard to say exactly which microbial groups are driving it. A more direct question: taken one at a time, is each genomic feature typically higher in low emitters, higher in high emitters, or about the same in both? Let's average each feature separately across the two groups and compare.

**What to expect:** a bar chart with two bars per genomic feature — one for the low-emitter average, one for the high-emitter average. For most of the 30 features, the two bars will be almost the same height: no real difference. But for a handful of features, you may be able to spot a visible gap between the blue and red bars. Keep an eye out for which ones those are.

**▶ Run the cell below**'''))


### Step 3: Compare average abundance by phenotype

In [ ]:
feature_means_low = X[labels == 0].mean(axis=0)
feature_means_high = X[labels == 1].mean(axis=0)

y_pos = np.arange(len(feature_names))
bar_height = 0.4

fig, ax = plt.subplots(figsize=(8, 10), dpi=100)
ax.barh(y_pos - bar_height / 2, feature_means_low, height=bar_height, color=LOW_COLOR, label='Low Emitter (average)')
ax.barh(y_pos + bar_height / 2, feature_means_high, height=bar_height, color=HIGH_COLOR, label='High Emitter (average)')
ax.set_yticks(y_pos)
ax.set_yticklabels(feature_names, fontsize=8)
ax.invert_yaxis()
ax.set_xlabel('Average Genomic Feature Value (relative to cohort average)')
ax.set_title('Average Relative Abundance per Genomic Feature, by Phenotype')
ax.legend()
plt.tight_layout()
plt.show()

display(Markdown('''> **Discussion:** Do any of the genomic features show a visible gap between the low- and high-emitter averages? Which ones — and does that match what you might have predicted before seeing the data?

### Now zoom in: the raw, individual values

The bar chart above shows *averages*, computed across dozens of samples — that averaging is what let any pattern show through. But no single animal is "average". Below is a histogram of every individual genomic feature value in the dataset, before any averaging, split by phenotype.

**What to expect:** the two distributions will overlap almost completely — from this view alone you cannot tell which numbers belong to which group. That's the real challenge: the biological signal is genuine, but at the level of a single sample it's subtle and spread thinly across many correlated features, not sitting in one obvious place.

**▶ Run the cell below**'''))


### Step 4: Zoom in on individual values

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5), dpi=100)
ax.hist(X[labels == 0].flatten(), bins=40, alpha=0.6, label='Low Emitter', color=LOW_COLOR, density=True)
ax.hist(X[labels == 1].flatten(), bins=40, alpha=0.6, label='High Emitter', color=HIGH_COLOR, density=True)
ax.set_xlabel('Genomic Feature Value (relative to cohort average)')
ax.set_ylabel('Density')
ax.set_title('Distribution of All 3,600 Genomic Feature Values, by Phenotype')
ax.legend()
plt.tight_layout()
plt.show()

display(Markdown('''As expected, the two distributions sit almost exactly on top of each other. Pooling all individual values together like this hides any structure — most of these numbers really are just noise, and the handful of genuinely informative features are diluted among them. To find the signal, we need to look at the data more carefully, starting with the individual samples themselves.

### What the data looks like

In reality, a dataset like this starts as raw DNA sequencing reads, which are first processed into **k-mer counts** and then summarised into the genomic embedding described in Section 1 — that processing happens in the lab and on a computing cluster, long before anyone sees a spreadsheet. What you see below is that finished genomic embedding: the starting point for *this* notebook, not raw sequencer output. Nothing has been transformed, reduced, or modelled any further than that. Each row is one animal, with:

- **sample_id** — a unique identifier for the animal
- **phenotype** — whether the animal is a `low_emitter` or `high_emitter`, based on its measured methane output
- **methane_g_per_day** — the animal's measured methane emissions, in grams per day
- **genomic feature columns** (e.g. `Butyrivibrio_cluster`, `Methanobrevibacter_cluster`) — one column per microbial genomic signature. Each value is a relative abundance score, scaled so that 0 is the **average across all 120 samples in this dataset** for that feature (much like a z-score). Positive means more abundant than the cohort average, negative means less abundant — the scale is relative to this group of animals, not an absolute biological unit

To keep the table readable we show only the first 10 of 120 samples, and the first 5 of 30 genomic features — but this is the full embedding, not a further summary or analysis of it.

**What to expect:** a plain data table. Scanning it by eye, the low and high emitters will **not** look obviously different — same story as the histogram above, just at the level of individual samples instead of pooled values.

**▶ Run the cell below**'''))


### Step 5: View the dataset

In [ ]:
sample_ids = [f'Sample_{i+1:03d}' for i in range(N)]

df = pd.DataFrame(X, columns=feature_names)
df.insert(0, 'sample_id', sample_ids)
df.insert(1, 'phenotype', np.where(labels == 0, 'low_emitter', 'high_emitter'))
df.insert(2, 'methane_g_per_day', np.round(methane, 1))

display_cols = ['sample_id', 'phenotype', 'methane_g_per_day'] + feature_names[:5]
display(df[display_cols].head(10))

display(Markdown('''As expected, nothing jumps out from scanning the raw numbers either — with 30 features per sample, the differences between groups are too spread out to see by eye. The next section starts making sense of this.

## Section 2: Visualising the Data — Can We See the Difference?

We've now looked at the raw data from several angles — composition, feature averages, a pooled histogram, a table of individual samples — and each time, low and high emitters looked essentially the same at a glance. That's because each sample has 30 numbers describing it — too many dimensions for a human eye to compare directly. **PCA (Principal Component Analysis)** is a technique built exactly for this problem: it compresses many dimensions into two, while preserving as much of the variation in the data as possible.

If low and high emitters really do differ in their microbiome, we would expect to see them separate when plotted in this 2D space. This is not a biological method — it is a mathematical one. We are simply asking: does the biology show up in the maths?

**What to expect:** a scatter plot with two colours — blue for low emitters, red for high emitters. They probably won't separate perfectly (this is noisy biological data), but you should be able to see a general trend of blue points sitting toward one side and red points toward the other.

**▶ Run the cell below**'''))


### Step 6: Visualise with PCA

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X)
var_explained = pca.explained_variance_ratio_ * 100

fig, ax = plt.subplots(figsize=(8, 5), dpi=100)
for label_val, name, color in [(0, 'Low Emitter', LOW_COLOR), (1, 'High Emitter', HIGH_COLOR)]:
    mask = labels == label_val
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1], c=color, label=name, s=50, alpha=0.75, edgecolor='white')

ax.set_xlabel(f'Principal Component 1 ({var_explained[0]:.1f}% variance explained)')
ax.set_ylabel(f'Principal Component 2 ({var_explained[1]:.1f}% variance explained)')
ax.set_title('PCA of Rumen Microbiome Genomic Embeddings')
ax.legend()
plt.tight_layout()
plt.show()

display(Markdown('''The scatter plot above showed some separation along PC1 — but PC1 itself is just a mathematical combination of all 30 original features, each weighted differently. To connect this back to biology, we can ask: which features contribute most to PC1? Those weights are called **loadings**, and the largest ones tell us what's actually driving the separation we just saw.

**What to expect:** a horizontal bar chart of the 10 features with the strongest pull on PC1. You should see methanogen-associated features (e.g. `Methanobrevibacter_cluster`) pointing one direction, and fibre-fermenting features (e.g. `Butyrivibrio_cluster`, `Fibrobacter_cluster`) pointing the other — matching the biology from Section 1.

**▶ Run the cell below**'''))


### Step 7: See what drives the separation

In [ ]:
loadings = pd.Series(pca.components_[0], index=feature_names)
top_loadings = loadings.reindex(loadings.abs().sort_values(ascending=False).index).head(10)

fig, ax = plt.subplots(figsize=(8, 5), dpi=100)
colors = [LOW_COLOR if v < 0 else HIGH_COLOR for v in top_loadings.values]
ax.barh(top_loadings.index[::-1], top_loadings.values[::-1], color=colors[::-1])
ax.set_xlabel('Loading on Principal Component 1')
ax.set_title('Top 10 Genomic Features Driving PC1')
plt.tight_layout()
plt.show()

display(Markdown('''> **Discussion:** Do the low and high emitters separate clearly? What would it mean if they overlapped completely?

> **Discussion:** Which microbial features appear to drive the separation? Do these make biological sense based on what you read in the introduction?

## Section 3: Training a Classifier — Can a Model Learn This?

PCA hinted that combining features mathematically can reveal some separation between the groups — but it only used 2 of the 30 dimensions, and it wasn't even trying to predict the phenotype, just summarise the variation. The natural next question: can a model that uses all 30 features, and is actually trained to predict the phenotype, do better?

A **classifier** is a mathematical model that learns to assign a label (low or high emitter) based on input data. We train it on some samples and test it on samples it has never seen — this tells us whether it has genuinely learned something useful, or simply memorised the training data.

We will use a **Random Forest**: an ensemble of many decision trees, each making simple yes/no splits on the features. You do not need to understand the maths — focus on what it learns and how well it performs.

**What to expect:** a printed accuracy somewhere in the 80–90% range — not 100%, because this is realistic, noisy biological data — followed by a confusion matrix that is mostly on the diagonal with a handful of misclassifications, and a small table of precision and recall for each class.

**▶ Run the cell below**'''))


### Step 8: Train a classifier

In [ ]:
X_train, X_test, y_train, y_test, methane_train, methane_test, idx_train, idx_test = train_test_split(
    X, labels, methane, np.arange(N), test_size=0.2, random_state=42, stratify=labels
)

clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

accuracy = accuracy_score(y_test, y_pred) * 100
print(f'Accuracy: {accuracy:.1f}%')

cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(8, 5), dpi=100)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Low Emitter', 'High Emitter'],
            yticklabels=['Low Emitter', 'High Emitter'], ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix: Random Forest Classifier')
plt.tight_layout()
plt.show()

precision, recall, f1, support = precision_recall_fscore_support(y_test, y_pred, labels=[0, 1])
metrics_df = pd.DataFrame({
    'Class': ['Low Emitter', 'High Emitter'],
    'Precision': np.round(precision, 2),
    'Recall': np.round(recall, 2),
})
display(metrics_df)

display(Markdown('''The accuracy and confusion matrix above tell us *how often* the model was right, but not *why* it got the rest wrong. A natural next question: are those mistakes scattered randomly, or do they happen for animals whose true methane output sits close to the boundary between "low" and "high"? Let's compare the actual methane values of the samples the model classified correctly against the ones it got wrong.

**What to expect:** two overlapping histograms of methane values — one for correctly classified samples, one for misclassified samples. If the model is doing something sensible, the misclassified samples should cluster around the middle of the methane range, where low and high emitters are genuinely hard to tell apart.

**▶ Run the cell below**'''))


### Step 9: Check where the model struggles

In [ ]:
correct = y_pred == y_test
plot_df = pd.DataFrame({
    'methane': methane_test,
    'correct': np.where(correct, 'Correctly Classified', 'Misclassified'),
})

fig, ax = plt.subplots(figsize=(8, 5), dpi=100)
for group, color in [('Correctly Classified', '#4CAF50'), ('Misclassified', '#9E9E9E')]:
    subset = plot_df[plot_df['correct'] == group]['methane']
    if len(subset) > 0:
        ax.hist(subset, bins=12, alpha=0.6, label=group, color=color)

ax.set_xlabel('Methane Emissions (g/day)')
ax.set_ylabel('Number of Test Samples')
ax.set_title('Methane Emissions: Correctly vs Incorrectly Classified Samples')
ax.legend()
plt.tight_layout()
plt.show()

display(Markdown('''> **Discussion:** The model is not perfect. What are the consequences of a false positive (predicting low emitter when the animal is actually high)? What about a false negative?

> **Discussion:** The model was trained on 96 samples. Do you think this is enough? What might change with 10,000 samples?

## Section 4: Explainability — Why Did the Model Decide That?

A model that makes accurate predictions is useful, but in science and in real-world applications we also need to know *why* it made a prediction. Random Forests provide a built-in measure called **feature importance** — which input features did the model rely on most heavily?

This is a simple form of **explainable AI (XAI)**: making the model's reasoning transparent. If the most important features make biological sense, that gives us more confidence in the model — and generates hypotheses for further research.

**What to expect:** a horizontal bar chart of the top 15 features by importance. The biologically relevant features (highlighted in orange) should tend to rank near the top — a sign the model has picked up on real biological signal rather than noise.

**▶ Run the cell below**'''))


### Step 10: Rank feature importance

In [ ]:
biological_features = set(low_boost.keys()) | set(high_boost.keys())

importances = pd.Series(clf.feature_importances_, index=feature_names)
top_importances = importances.sort_values(ascending=False).head(15)

colors = ['#FF9800' if f in biological_features else '#B0BEC5' for f in top_importances.index]

fig, ax = plt.subplots(figsize=(8, 5), dpi=100)
ax.barh(top_importances.index[::-1], top_importances.values[::-1], color=colors[::-1])
ax.set_xlabel('Feature Importance')
ax.set_title('Top 15 Features by Random Forest Importance')

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#FF9800', label='Biologically relevant feature'),
                    Patch(facecolor='#B0BEC5', label='Other feature')]
ax.legend(handles=legend_elements, loc='lower right')
plt.tight_layout()
plt.show()

display(Markdown('''The chart above shows which features matter most *overall*, averaged across every prediction the model makes — but it doesn't tell us what happened for any one animal. To make this concrete, let's zoom into two individual test samples — one correctly predicted as a low emitter, one as a high emitter — and see exactly which features tipped the model's decision for each of them.

**What to expect:** two side-by-side bar charts — one for a correctly classified low emitter, one for a correctly classified high emitter — each showing the 5 features that contributed most to that individual prediction.

**▶ Run the cell below**'''))


### Step 11: Explain individual predictions

In [ ]:
low_idx = [i for i, (p, c) in enumerate(zip(y_pred, correct)) if p == 0 and c][0]
high_idx = [i for i, (p, c) in enumerate(zip(y_pred, correct)) if p == 1 and c][0]

def top_features_for_sample(sample_row, importances, n=5):
    contribution = sample_row * importances.values
    contrib_series = pd.Series(contribution, index=importances.index)
    return contrib_series.reindex(contrib_series.abs().sort_values(ascending=False).index).head(n)

low_sample_contrib = top_features_for_sample(X_test[low_idx], importances)
high_sample_contrib = top_features_for_sample(X_test[high_idx], importances)

fig, axes = plt.subplots(1, 2, figsize=(12, 5), dpi=100)

axes[0].barh(low_sample_contrib.index[::-1], low_sample_contrib.values[::-1], color=LOW_COLOR)
axes[0].set_title('Why the model said LOW EMITTER')
axes[0].set_xlabel('Contribution (feature value x importance)')

axes[1].barh(high_sample_contrib.index[::-1], high_sample_contrib.values[::-1], color=HIGH_COLOR)
axes[1].set_title('Why the model said HIGH EMITTER')
axes[1].set_xlabel('Contribution (feature value x importance)')

plt.tight_layout()
plt.show()

display(Markdown('''> **Discussion:** The model identified Methanobrevibacter as a key predictor of high emissions. Does this mean Methanobrevibacter *causes* high emissions? How would you test this?

> **Discussion:** If a model is accurate but its top features make no biological sense, what would you conclude?

## Section 5: What This Means — and What It Doesn't

We have demonstrated that a genomic representation of a microbial community carries **predictive signal** for a host phenotype, and that a simple machine learning model can extract and explain that signal in a way that aligns with known biology.

What this does **not** prove is causality, generalisability across different breeds, diets, or environments, or readiness for use as a real-world management tool. Real microbiome research faces further challenges: small sample sizes, **batch effects** across different studies, **compositional data** constraints (relative abundances that must sum to a fixed total), and the difficulty of integrating multiple independent datasets.

This is exactly the kind of question that motivated research programmes such as James Barnard's PhD — using explainable AI to understand which microbial drivers are genuinely robust across datasets, rather than artefacts of a single study.

## Section 6: Summary and Key Concepts

- **Genomic embeddings**: k-mer frequency vectors used as numerical representations of microbial communities
- **Dimensionality reduction (PCA)**: compressing many features into a small number of axes that preserve variation, for visualisation
- **Classification (Random Forest)**: an ensemble model that learns to assign labels from data, evaluated on held-out test samples
- **Feature importance**: a simple form of explainable AI (XAI) that shows which inputs a model relied on
- **Prediction vs causal explanation**: a model can predict accurately without proving *why* something happens biologically

**Further Reading**
- Roehe, R. et al. (2016) "Bovine Host Genetic Variation Influences Rumen Microbial Methane Production with Best Selection Criterion for Low Methane Emitting and Efficiently Feed Converting Hosts Based on Metagenomic Gene Abundance." *PLOS Genetics*.
- Ritchie, H. (2020) "Emissions from food alone could use up all of our budget for 1.5°C or 2°C — but this isn't the most likely outcome." Our World in Data. https://ourworldindata.org/food-emissions-carbon-budget
- Lundberg, S. M. & Lee, S.-I. (2017) "A Unified Approach to Interpreting Model Predictions." *NeurIPS*. (Background on explainable AI / feature attribution.)'''))
